# TT Ball Position Regressor — Fine-tune MobileNetV3-Small

Train a model to detect table tennis ball position from a single frame.

**Input:** 320×320 image  
**Output:** (x, y, confidence) — normalized ball position + presence

## How to use:
1. Open this notebook in Colab (browser): File → Upload notebook
2. Set runtime to GPU: Runtime → Change runtime type → T4 GPU
3. Upload `data_regressor.zip` via the Files sidebar (folder icon on the left)
4. Run all cells in order

## Cell 1: Load data from Google Drive

1. Upload `data_regressor.zip` to the **root** of your Google Drive at drive.google.com
2. Run this cell — it will pop up a sign-in window

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

# Copy from Drive to local storage (faster I/O during training)
!cp /content/drive/MyDrive/data_regressor.zip /content/
!unzip -q /content/data_regressor.zip -d /content/

print("Contents:")
!ls /content/data_regressor/train/
!wc -l /content/data_regressor/train/labels.csv
!wc -l /content/data_regressor/val/labels.csv

## Cell 2: Dataset with on-the-fly augmentation

In [ ]:
import csv
import random
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

IMG_SIZE = 320

# ImageNet normalization (required for pretrained MobileNetV3)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class BallDataset(Dataset):
    """Loads 320x320 images + (x, y, confidence) labels."""

    def __init__(self, root_dir: str, augment: bool = False):
        self.img_dir = Path(root_dir) / "images"
        self.augment = augment
        self.samples = []

        csv_path = Path(root_dir) / "labels.csv"
        with open(csv_path) as f:
            reader = csv.DictReader(f)
            for row in reader:
                self.samples.append({
                    "file": row["filename"],
                    "x": float(row["x"]),
                    "y": float(row["y"]),
                    "conf": float(row["confidence"]),
                })

        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)

        # Augmentation transforms
        self.color_jitter = transforms.ColorJitter(
            brightness=0.15, contrast=0.15, saturation=0.1, hue=0.02
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(self.img_dir / s["file"]).convert("RGB")
        x, y, conf = s["x"], s["y"], s["conf"]

        if self.augment:
            # Random horizontal flip
            if random.random() > 0.5:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
                if conf > 0:
                    x = 1.0 - x

            # Color jitter
            img = self.color_jitter(img)

        img_t = self.normalize(self.to_tensor(img))
        label = torch.tensor([x, y, conf], dtype=torch.float32)
        return img_t, label


# Create datasets and dataloaders
train_ds = BallDataset("/content/data_regressor/train", augment=True)
val_ds = BallDataset("/content/data_regressor/val", augment=False)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_ds)} samples, {len(val_loader)} batches")

## Cell 3: Model — MobileNetV3-Small with regression head

In [ ]:
import torch.nn as nn
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights


class BallRegressor(nn.Module):
    """MobileNetV3-Small backbone + 3-output regression head (x, y, confidence)."""

    def __init__(self):
        super().__init__()
        # Load pretrained backbone
        backbone = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)

        # Everything except the classifier
        self.features = backbone.features
        self.avgpool = backbone.avgpool

        # New regression head: 576 features → 3 outputs
        self.head = nn.Sequential(
            nn.Linear(576, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 3),  # x, y, confidence
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = x.flatten(1)
        x = self.head(x)
        # Sigmoid to clamp outputs to 0-1
        return torch.sigmoid(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BallRegressor().to(device)

# Count parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Device: {device}")
print(f"Parameters: {total:,} total, {trainable:,} trainable")
print(f"Model size: ~{total * 4 / 1024 / 1024:.1f} MB (float32)")

## Cell 4: Loss function

In [ ]:
import torch.nn.functional as F


def ball_loss(pred, target):
    """
    Combined loss for ball position regression.

    pred:   (batch, 3) — predicted [x, y, confidence] (all sigmoid, 0-1)
    target: (batch, 3) — ground truth [x, y, confidence]

    - MSE on (x, y) only for frames where ball is present (conf == 1)
    - BCE on confidence for ALL frames
    """
    pred_xy = pred[:, :2]
    pred_conf = pred[:, 2]
    true_xy = target[:, :2]
    true_conf = target[:, 2]

    # Confidence loss: all frames
    conf_loss = F.binary_cross_entropy(pred_conf, true_conf)

    # Position loss: only ball-present frames
    ball_mask = true_conf > 0.5
    if ball_mask.any():
        pos_loss = F.mse_loss(pred_xy[ball_mask], true_xy[ball_mask])
    else:
        pos_loss = torch.tensor(0.0, device=pred.device)

    # Weight position loss higher — it's the main task
    return 2.0 * pos_loss + conf_loss


print("Loss function ready.")

## Cell 5: Train

In [ ]:
import math

EPOCHS = 50
FREEZE_EPOCHS = 5  # freeze backbone for first N epochs

# Phase 1: freeze backbone, train only the head
for param in model.features.parameters():
    param.requires_grad = False

optimizer = torch.optim.Adam(model.head.parameters(), lr=1e-3)
scheduler = None  # no schedule for warmup phase

best_val_loss = float("inf")
history = {"train_loss": [], "val_loss": []}

for epoch in range(1, EPOCHS + 1):
    # Unfreeze backbone after warmup
    if epoch == FREEZE_EPOCHS + 1:
        print(f"\n--- Unfreezing backbone at epoch {epoch} ---\n")
        for param in model.features.parameters():
            param.requires_grad = True
        # New optimizer with lower LR for backbone
        optimizer = torch.optim.Adam([
            {"params": model.features.parameters(), "lr": 1e-4},
            {"params": model.head.parameters(), "lr": 5e-4},
        ])
        # Cosine schedule for remaining epochs
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS - FREEZE_EPOCHS, eta_min=1e-6
        )

    # ── Train ──
    model.train()
    train_loss_sum = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        pred = model(imgs)
        loss = ball_loss(pred, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item() * imgs.size(0)

    train_loss = train_loss_sum / len(train_ds)

    # ── Validate ──
    model.eval()
    val_loss_sum = 0.0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            pred = model(imgs)
            loss = ball_loss(pred, labels)
            val_loss_sum += loss.item() * imgs.size(0)

    val_loss = val_loss_sum / len(val_ds)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    if scheduler:
        scheduler.step()

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "/content/best_model.pth")
        marker = " \u2605"
    else:
        marker = ""

    lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch:3d}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}  lr={lr:.1e}{marker}")

print(f"\nBest val loss: {best_val_loss:.4f}")
# Load best weights
model.load_state_dict(torch.load("/content/best_model.pth"))
print("Loaded best model weights.")

## Cell 6: Evaluate

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── Loss curves ──
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(history["train_loss"], label="train")
ax.plot(history["val_loss"], label="val")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.set_title("Training Loss")
plt.tight_layout()
plt.show()

# ── Metrics on val set ──
model.eval()
pixel_errors = []
conf_correct = 0
conf_total = 0
ball_total = 0
ball_detected = 0
noball_total = 0
noball_correct = 0

with torch.no_grad():
    for imgs, labels in val_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        pred = model(imgs)

        for i in range(imgs.size(0)):
            true_conf = labels[i, 2].item()
            pred_conf = pred[i, 2].item()
            conf_total += 1

            if true_conf > 0.5:
                # Ball present
                ball_total += 1
                if pred_conf > 0.5:
                    ball_detected += 1
                    # Pixel distance
                    dx = (pred[i, 0].item() - labels[i, 0].item()) * IMG_SIZE
                    dy = (pred[i, 1].item() - labels[i, 1].item()) * IMG_SIZE
                    dist = math.sqrt(dx * dx + dy * dy)
                    pixel_errors.append(dist)
                    conf_correct += 1
            else:
                # No ball
                noball_total += 1
                if pred_conf <= 0.5:
                    noball_correct += 1
                    conf_correct += 1

pixel_errors = np.array(pixel_errors)
print(f"=== Validation Results ===")
print(f"Confidence accuracy: {conf_correct}/{conf_total} = {100*conf_correct/conf_total:.1f}%")
print(f"  Ball recall:    {ball_detected}/{ball_total} = {100*ball_detected/max(ball_total,1):.1f}%")
print(f"  No-ball accuracy: {noball_correct}/{noball_total} = {100*noball_correct/max(noball_total,1):.1f}%")
print()
if len(pixel_errors) > 0:
    print(f"Position error (px on {IMG_SIZE}x{IMG_SIZE}):")
    print(f"  Mean:   {pixel_errors.mean():.1f} px")
    print(f"  Median: {np.median(pixel_errors):.1f} px")
    print(f"  P90:    {np.percentile(pixel_errors, 90):.1f} px")
    print(f"  Max:    {pixel_errors.max():.1f} px")

    # Histogram
    fig, ax = plt.subplots(1, 1, figsize=(8, 3))
    ax.hist(pixel_errors, bins=30, color="steelblue", edgecolor="white")
    ax.axvline(pixel_errors.mean(), color="red", linestyle="--", label=f"mean={pixel_errors.mean():.1f}")
    ax.set_xlabel("Pixel error")
    ax.set_ylabel("Count")
    ax.set_title("Position Error Distribution")
    ax.legend()
    plt.tight_layout()
    plt.show()

## Cell 7: Export to TFLite

In [ ]:
!pip install -q onnx onnx2tf tensorflow-cpu flatbuffers

import onnx

# Step 1: PyTorch → ONNX
model.eval()
model.cpu()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
onnx_path = "/content/ball_regressor.onnx"

torch.onnx.export(
    model, dummy, onnx_path,
    input_names=["image"],
    output_names=["output"],
    dynamic_axes={"image": {0: "batch"}, "output": {0: "batch"}},
    opset_version=13,
)
print(f"ONNX exported: {onnx_path}")
print(f"ONNX size: {os.path.getsize(onnx_path) / 1024 / 1024:.1f} MB")

# Step 2: ONNX → TFLite (with INT8 quantization)
import subprocess
tflite_path = "/content/ball_regressor.tflite"

# Use onnx2tf for conversion
result = subprocess.run(
    ["onnx2tf", "-i", onnx_path, "-o", "/content/tflite_out",
     "-oiqt",  # INT8 quantization
     "-qt", "per-tensor"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("onnx2tf failed, trying float16 fallback...")
    # Fallback: float16 via TensorFlow Lite converter
    import tensorflow as tf
    import onnx
    from onnx_tf.backend import prepare

    onnx_model = onnx.load(onnx_path)
    tf_rep = prepare(onnx_model)
    tf_rep.export_graph("/content/tf_saved_model")

    converter = tf.lite.TFLiteConverter.from_saved_model("/content/tf_saved_model")
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    tflite_model = converter.convert()

    with open(tflite_path, "wb") as f:
        f.write(tflite_model)
else:
    # Find the quantized tflite in output
    import glob
    import shutil
    tflite_files = glob.glob("/content/tflite_out/*.tflite")
    if tflite_files:
        shutil.copy(tflite_files[0], tflite_path)

print(f"\nTFLite model: {tflite_path}")
print(f"TFLite size: {os.path.getsize(tflite_path) / 1024 / 1024:.1f} MB")

## Cell 8: Download model

In [ ]:
# Download both the PyTorch weights and TFLite model
from google.colab import files

# TFLite for Android
files.download("/content/ball_regressor.tflite")

# PyTorch weights (backup, for further training later)
files.download("/content/best_model.pth")

print("Done! Copy ball_regressor.tflite to:")
print("  app/src/main/assets/ball_regressor.tflite")